### **Setting Up Your Workspace**

In [1]:
# src/feature_engineering_advanced.py
import pandas as pd
import pandas_ta as ta
import pytz
from datetime import datetime, time, timedelta

print("--- Starting Advanced Feature Engineering ---")

--- Starting Advanced Feature Engineering ---


In [2]:
def calculate_all_indicators(df):
    """Calculates a rich set of technical indicators on the DataFrame."""
    print("Calculating ~20 technical indicators...")
    
    # Use the pandas_ta Strategy builder for efficiency
    # We will list the indicators mentioned in the paper.
    # Note: Some names might be slightly different in pandas_ta.
    # 'kind' is the indicator name, you can find them in the pandas_ta documentation.
    my_study = ta.Study(
        name="RL Paper Indicators",
        description="A collection of ~20 indicators from the RL paper",
        ta=[
            # Momentum Indicators
            {"kind": "rsi"},          # Relative Strength Index
            {"kind": "mom"},          # Momentum
            {"kind": "stoch"},        # Stochastic Oscillator (%K and %D)
            {"kind": "macd"},         # Moving Average Convergence Divergence
            {"kind": "cci"},          # Commodity Channel Index
            {"kind": "roc"},          # Rate of Change
            {"kind": "cmo"},          # Chande Momentum Oscillator
            {"kind": "stochrsi"},     # Stochastic RSI
            {"kind": "willr"},        # Williams %R (similar to Ultimate Oscillator)
            
            # Trend Indicators
            {"kind": "adx"},          # Average Directional Movement Index
            {"kind": "trix"},         # TRIX
            {"kind": "psar"},         # Parabolic SAR
            {"kind": "tema"},         # Triple Exponential Moving Average
            {"kind": "trima"},        # Triangular Moving Average
            {"kind": "wma"},          # Weighted Moving Average
            {"kind": "dema"},         # Double Exponential Moving Average
            
            # Volume and Volatility Indicators
            {"kind": "mfi"},          # Money Flow Index
            {"kind": "bop"},          # Balance of Power
            {"kind": "atr"},          # Average True Range
        ]
    )
    
    # Run the strategy on the DataFrame (this appends all columns)
    df.ta.study(my_study)
    
    return df

### **Step 1: Load the Data and Make it Timezone-Aware**

Think of your raw data from MT5 as having a "naive" timestamp. It's just a number. We need to give it a "birth certificate" that says, "This time is in the UTC timezone." This is the most important first step.

In [3]:
# --- Step 1: Load Data and Set Timezone ---
print('Loading raw data...')
year = 2018
# Load the hourly data downloaded earlier using fastparquet(pip install this to allow pandas to use for the data loading)
df_raw = pd.read_parquet(f'../data/raw/xauusd_h1_{year}_present.parquet')

# The 'time' column is our master key. We make it the index of the Dataframe.
df_raw.set_index('time', inplace=True)


"""
Localize the naive index to UTC

This is to deal with forex timezones, standard timezone and Daylight Saving Time(DST) timezones affecting the London & New York time.

CRITICAL: Tell pandas that the existing timestamps are in UTC.
This doesn't change the numbers, it just adds the context.
"""
# --- Rename volume column BEFORE calculating indicators ---
df_raw.rename(columns={'tick_volume': 'volume'}, inplace=True)
df = df_raw.tz_localize('UTC')



print(f"Data loaded successfully. Index timezone is now: {df.index.tz}")


Loading raw data...
Data loaded successfully. Index timezone is now: UTC


### **2. Pre-calculate technical indicators**

It's much faster to calculate indicators like RSI and ATR on the entire dataset at once, rather than re-calculating them inside our loop every day. We'll use the pandas_ta library for this.

In [4]:
# --- Step 2: Pre-calculate Technical Indicators ---
print("\nStep 2: Calculating technical indicators...")

# --- Pre-calculate ALL indicators ---
df = calculate_all_indicators(df)

# --- NEW SECTION: Handle Missing Indicator Values ---
print("Handling missing values from indicators...")

# 1. Specifically handle the PSAR columns by merging them.
# Find the exact column names by printing df.columns after calculating. They might have suffixes.
# Let's assume the columns are 'PSARl_0.02_0.2' and 'PSARs_0.02_0.2'
psar_long_col = 'PSARl_0.02_0.2'
psar_short_col = 'PSARs_0.02_0.2'

if psar_long_col in df.columns and psar_short_col in df.columns:
    # Create a single 'PSAR' column. Where 'PSARl' is NaN, it will use the value from 'PSARs'.
    df['PSAR'] = df[psar_long_col].fillna(df[psar_short_col])
    # Now we can drop the original two columns
    df.drop(columns=[psar_long_col, psar_short_col], inplace=True)
    print("Merged PSARl and PSARs into a single 'PSAR' column.")

# 2. Handle Stochastic Oscillator (%K and %D). They often have NaNs at the start.
# pandas_ta creates 'STOCHk_14_3_3' and 'STOCHd_14_3_3'.
# Let's check for other similar multi-column indicators if needed.

# 3. Apply a general forward-fill for all remaining NaNs.
# This carries the last valid observation forward. It's the standard for time-series.
df.ffill(inplace=True)
print("Applied forward-fill (ffill) to remaining NaNs.")


Step 2: Calculating technical indicators...
Calculating ~20 technical indicators...
Handling missing values from indicators...
Merged PSARl and PSARs into a single 'PSAR' column.
Applied forward-fill (ffill) to remaining NaNs.


In [7]:
print("Indicators calculated. New columns added.")
df.tail() # Use .tail() to see the latest calculated values

Indicators calculated. New columns added.


,open,high,low,close,volume,spread,real_volume,RSI_14,MOM_10,STOCHk_14_3_3,...,PSARaf_0.02_0.2,PSARr_0.02_0.2,TEMA_10,TRIMA_10,WMA_10,DEMA_10,MFI_14,BOP,ATRr_14,PSAR
time,,,,,,,,,,,,,,,,,,,,,
2025-10-13 02:00:00+00:00,4042.20,4060.29,4032.32,4035.10,5864,5,0,64.019931,47.54,84.639998,...,0.1,0,4034.557269,3998.976111,4013.493455,4025.794290,59.279563,-0.253843,21.842119,3983.985606
2025-10-13 03:00:00+00:00,4034.69,4060.03,4024.47,4043.85,6291,5,0,66.363408,63.24,83.263275,...,0.1,0,4042.982452,4004.957500,4020.708727,4034.026685,59.998391,0.257593,22.821968,3991.616045
2025-10-13 04:00:00+00:00,4043.78,4054.90,4036.03,4053.06,6203,5,0,68.676065,63.57,81.814798,...,0.1,0,4052.080440,4012.120556,4028.448727,4042.906993,67.182570,0.491786,22.539684,3998.483441
2025-10-13 05:00:00+00:00,4052.97,4055.81,4041.88,4047.96,5141,8,0,65.971180,50.05,86.601161,...,0.1,0,4055.090287,4019.441111,4034.105636,4047.501348,73.644022,-0.359655,21.924707,4004.664097
2025-10-13 06:00:00+00:00,4048.00,4056.47,4046.34,4051.90,4790,5,0,67.050868,68.36,89.597290,...,0.1,0,4058.104672,4027.224167,4039.568909,4051.894550,79.818944,0.384995,21.082228,4010.226687


### **Step 3: The Main Loop - The Heart of the Script**

Now we'll go through our data day by day. For each day, we'll perform our session logic.

In [8]:
# --- Step 3: Loop Through Each Day to Engineer Features ---
print("\nStep 3: Starting daily feature engineering loop...")

# This list will hold the dictionary for each day's calculated data.
daily_data_list = []

# Define the local timezones we need.
london_tz = pytz.timezone('Europe/London')
# ny_tz = pytz.timezone("America/New_York") # for New York


# df.index.date gives us just the date part (e.g., 2025-09-20)
# We group by this to process one day at a time.
for day in df.index.normalize().unique():
    try:
        # Define session boundaries
        london_open_local = london_tz.localize(datetime.combine(day, time(8, 0)))
        london_close_local = london_tz.localize(datetime.combine(day, time(17, 0)))
        london_open_utc = london_open_local.astimezone(pytz.utc)
        london_close_utc = london_close_local.astimezone(pytz.utc)
        
        previous_day = day - timedelta(days=1)
        asia_part1 = df.loc[str(previous_day.date())].between_time('22:00', '23:59')
        asia_part2 = df.loc[str(day.date())].between_time('00:00', '07:59')
        asia_session = pd.concat([asia_part1, asia_part2])
        
        london_session = df.loc[(df.index >= london_open_utc) & (df.index < london_close_utc)]
        
        if asia_session.empty or london_session.empty:
            continue

        # --- Feature Calculation ---
        asia_close_price = asia_session['close'].iloc[-1]
        end_of_asia_ts = asia_session.index[-1]
        
        # Get the values of all our new indicators at the end of the Asian session
        indicator_values = df.loc[end_of_asia_ts]

        # --- Target Calculation ---
        london_open = london_session['open'].iloc[0]
        london_close = london_session['close'].iloc[-1]
        london_direction = 1 if london_close > london_open else 0
        london_return = (london_close - london_open) / london_open

        # Create the feature dictionary for this day
        # We start with our original features and add the new ones
        feature_dict = {
            'date': day.date(),
            'day_of_week': day.dayofweek,
            'asia_return': (asia_close_price - asia_session['open'].iloc[0]) / asia_session['open'].iloc[0],
            'asia_range': asia_session['high'].max() - asia_session['low'].min(),
            'london_direction': london_direction,
            'london_return': london_return
        }

        # Add all the indicator values to our dictionary
        # We'll grab the relevant indicator columns (you can find their exact names by printing df.columns)
        # For simplicity, we grab all columns generated by the strategy
        indicator_cols = [col for col in df.columns if col.startswith(('RSI', 'MOM', 'STOCH', 'MACD', 'CCI', 'ROC', 'CMO', 'WILLR', 'ADX', 'TRIX', 'PSAR', 'TEMA', 'TRIMA', 'WMA', 'DEMA', 'MFI', 'BOP', 'ATRr'))]
        for col in indicator_cols:
            # We add a check here in case an indicator column is all NaNs at the start
            if pd.notna(indicator_values[col]):
                feature_dict[col] = indicator_values[col]

        daily_data_list.append(feature_dict)
        
        
    except Exception as e:
        # If anything goes wrong for a specific day, print it and continue.
        print(f"Could not process {day.date()}: {e}")
        continue # Move to the next day
    
print(f"Loop finished. Processed {len(daily_data_list)} trading days.")


Step 3: Starting daily feature engineering loop...
Could not process 2018-01-02: '2018-01-01'
Loop finished. Processed 2007 trading days.


### **Step 4: Final Assembly and Saving**
The final step is to convert our list of dictionaries into a clean, final DataFrame and save it.

In [9]:
len(daily_data_list)

2007

In [10]:
print("Assembling and saving final ADVANCED feature DataFrame...")
final_df = pd.DataFrame(daily_data_list)
final_df['date'] = pd.to_datetime(final_df['date'])
final_df.set_index('date', inplace=True)
final_df.dropna(inplace=True) # Drop rows with NaNs from indicator warm-up

output_path = '../data/processed/hyp_a_features_advanced.parquet'
final_df.to_parquet(output_path)
print(f"Successfully saved ADVANCED feature data to {output_path}")
final_df.info()

Assembling and saving final ADVANCED feature DataFrame...
Successfully saved ADVANCED feature data to ../data/processed/hyp_a_features_advanced.parquet
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2006 entries, 2018-01-04 to 2025-10-10
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   day_of_week          2006 non-null   int64  
 1   asia_return          2006 non-null   float64
 2   asia_range           2006 non-null   float64
 3   london_direction     2006 non-null   int64  
 4   london_return        2006 non-null   float64
 5   RSI_14               2006 non-null   float64
 6   MOM_10               2006 non-null   float64
 7   STOCHk_14_3_3        2006 non-null   float64
 8   STOCHd_14_3_3        2006 non-null   float64
 9   STOCHh_14_3_3        2006 non-null   float64
 10  CCI_14_0.015         2006 non-null   float64
 11  ROC_10               2006 non-null   float64
 12  CMO_14            

In [11]:
final_df

,day_of_week,asia_return,asia_range,london_direction,london_return,RSI_14,MOM_10,STOCHk_14_3_3,STOCHd_14_3_3,STOCHh_14_3_3,...,DEMA_10,MFI_14,BOP,ATRr_14,PSAR,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,TRIX_30_9,TRIXs_30_9
date,,,,,,,,,,,,,,,,,,,,,
2018-01-04,3,-0.002217,10.84,1,0.003565,45.078922,-6.11,22.171226,15.319033,6.852193,...,1307.859555,30.587346,0.779851,2.596249,1315.105707,-1.658247,-0.679946,-0.978301,-0.001497,-0.000003
2018-01-05,4,-0.003126,6.02,0,-0.003038,55.991011,1.03,62.873276,68.619618,-5.746342,...,1322.031673,77.862940,-0.853535,2.453258,1325.527730,2.172012,-0.035823,2.207835,0.005415,0.002398
2018-01-08,0,-0.002128,4.77,1,0.000759,45.015562,-2.39,59.471082,69.017323,-9.546241,...,1319.590210,65.308675,-0.721854,2.246994,1314.205989,0.393206,-0.220134,0.613340,0.008113,0.008157
2018-01-09,1,0.000212,4.95,0,-0.007216,51.761549,0.41,66.047107,63.291493,2.755614,...,1318.833379,61.996261,0.402878,1.891777,1315.422784,-0.006538,0.033848,-0.040386,0.003413,0.004123
2018-01-10,2,-0.003075,6.25,1,0.006139,36.431761,-3.03,28.646518,36.936254,-8.289736,...,1310.009542,46.961478,-0.506173,2.116085,1312.483658,-1.662555,-0.042516,-1.620039,-0.007136,-0.005239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-06,0,0.012910,56.02,1,0.001191,77.465799,56.69,96.256108,92.696780,3.559328,...,3929.823490,64.533584,0.809028,12.148528,3905.487968,16.711563,5.015355,11.696208,0.014167,0.009970
2025-10-07,1,0.004472,21.58,1,0.001518,72.026423,18.11,87.543545,77.652362,9.891183,...,3973.098137,63.948098,0.469274,11.827424,3956.424691,15.175529,-0.842409,16.017938,0.043140,0.039891
2025-10-08,2,0.011485,48.33,1,0.006244,77.371131,45.83,98.762390,97.495532,1.266858,...,4016.236540,84.320633,0.964869,11.980454,3988.325533,15.659648,3.251609,12.408038,0.042414,0.042066
